# Notebook 14: Gradio and HuggingFace Spaces

**Learning Objectives:**
- Build interactive ML demos with Gradio
- Create UIs for text, image, and audio models
- Use Gradio Blocks for custom layouts
- Understand how to deploy demos to HuggingFace Spaces

**Prerequisites:** Notebooks 01–09 (basic HuggingFace pipeline familiarity)

## Prerequisites

### Hardware Requirements

| Requirement | Specification |
|-------------|---------------|
| **CPU** | Any modern CPU |
| **RAM** | 4GB minimum, 8GB recommended |
| **Storage** | 2GB free (for model downloads) |
| **GPU** | Optional (CPU models used by default) |

### Software Requirements
- Python 3.8+
- Libraries: `gradio`, `transformers`, `torch`, `Pillow`
- See `requirements.txt` for full list

### Installation
```bash
pip install gradio
```

## Expected Behaviors

### Gradio Demos
- Each `demo.launch()` call starts a local web server
- In Jupyter, the demo appears as an embedded iframe below the cell
- A public URL is available when `share=True` (valid for 72 hours)
- Close demos by interrupting the cell or calling `demo.close()`

### Common Output
```
Running on local URL:  http://127.0.0.1:7860
Running on public URL: https://xxxxx.gradio.live
```

### Common Issues
- **Port conflict**: If port 7860 is in use, Gradio auto-selects another
- **Demo not showing**: Restart kernel and re-run; only one demo can run per cell
- **Slow first inference**: Model downloads on first use (subsequent calls are fast)
- **Share link not working**: Some corporate networks block Gradio tunnels

## Overview

**Gradio** is a Python library for building interactive web interfaces for ML models. In just a few lines of code, you can create a shareable demo that lets anyone interact with your model through a browser.

**Why Gradio Matters:**
- **Show, don't tell**: A demo is worth a thousand README lines
- **Rapid prototyping**: Test model behavior interactively before writing evaluation code
- **Sharing**: Generate a public link for collaborators, instructors, or recruiters
- **HuggingFace Spaces**: Deploy permanent, free demos on the HuggingFace Hub

**What We'll Build:**
1. A text sentiment analyzer
2. An image classifier
3. A multi-model comparison tool using Blocks

## Setup and Installation

In [ ]:
# Import libraries
import random
import sys
import numpy as np
import torch
import transformers
from transformers import pipeline
import warnings
warnings.filterwarnings('ignore')

# Check for Gradio
try:
    import gradio as gr
    GRADIO_AVAILABLE = True
except ImportError:
    GRADIO_AVAILABLE = False
    print("Gradio not installed. Install with: pip install gradio")

# Set seed for reproducibility
SEED = 1103
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Device selection
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Version metadata
print(f"Python: {sys.version.split()[0]}")
print(f"PyTorch: {torch.__version__}")
print(f"Transformers: {transformers.__version__}")
if GRADIO_AVAILABLE:
    print(f"Gradio: {gr.__version__}")
if torch.cuda.is_available():
    print(f"CUDA: {torch.version.cuda}")
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## Part 1: Gradio Basics

The simplest Gradio demo needs three things: an input component, an output component, and a Python function that connects them. The `gr.Interface` class wraps these into a complete web app with a single line.

In [ ]:
# The simplest possible Gradio demo
def greet(name: str) -> str:
    """Return a greeting message.

    Args:
        name: The name to greet.

    Returns:
        A greeting string.
    """
    return f"Hello, {name}! Welcome to Gradio."


if GRADIO_AVAILABLE:
    demo = gr.Interface(
        fn=greet,
        inputs=gr.Textbox(label="Your Name", placeholder="Enter your name..."),
        outputs=gr.Textbox(label="Greeting"),
        title="Hello World Demo",
        description="A simple Gradio demo that greets you by name.",
    )
    demo.launch()
else:
    print("Skipping: Gradio not available.")

### Input and Output Components

Gradio provides specialized components for different data types. Each component handles the UI rendering, data conversion, and validation automatically. Here are the most common ones used with ML models.

In [ ]:
if GRADIO_AVAILABLE:
    # Show available component types
    components = {
        "Component": [
            "gr.Textbox", "gr.Number", "gr.Slider",
            "gr.Checkbox", "gr.Dropdown", "gr.Image",
            "gr.Audio", "gr.File", "gr.Label",
        ],
        "Use Case": [
            "Text input/output", "Numeric values", "Range selection",
            "Boolean toggle", "Category selection", "Image upload/display",
            "Audio upload/playback", "File upload", "Classification labels",
        ],
        "ML Example": [
            "Text generation prompt", "Temperature parameter", "Max tokens slider",
            "Use GPU toggle", "Model selection", "Image classification input",
            "Speech recognition input", "Document upload", "Sentiment prediction",
        ],
    }

    import pandas as pd
    print("=== GRADIO COMPONENTS ===")
    print(pd.DataFrame(components).to_string(index=False))
else:
    print("Skipping: Gradio not available.")

## Part 2: Building an NLP Demo

Let's build a sentiment analysis demo using a HuggingFace pipeline. The `gr.Label` component is perfect for classification tasks — it displays predicted labels with confidence bars, giving users an intuitive view of model predictions.

In [ ]:
# Load sentiment analysis model
SENTIMENT_MODEL = "distilbert-base-uncased-finetuned-sst-2-english"
sentiment_classifier = pipeline("sentiment-analysis", model=SENTIMENT_MODEL, device=-1)
print(f"Loaded: {SENTIMENT_MODEL}")

In [ ]:
# Build sentiment analysis demo
def analyze_sentiment(text: str) -> dict[str, float]:
    """Classify text sentiment and return label confidences.

    Args:
        text: Input text to classify.

    Returns:
        Dictionary mapping sentiment labels to confidence scores.
    """
    if not text.strip():
        return {"POSITIVE": 0.0, "NEGATIVE": 0.0}
    result = sentiment_classifier(text)[0]
    label = result['label']
    score = result['score']
    # Return both labels with their scores
    if label == "POSITIVE":
        return {"POSITIVE": score, "NEGATIVE": 1 - score}
    else:
        return {"NEGATIVE": score, "POSITIVE": 1 - score}


if GRADIO_AVAILABLE:
    sentiment_demo = gr.Interface(
        fn=analyze_sentiment,
        inputs=gr.Textbox(
            label="Enter Text",
            placeholder="Type a movie review, product review, or any text...",
            lines=3,
        ),
        outputs=gr.Label(label="Sentiment", num_top_classes=2),
        title="Sentiment Analysis Demo",
        description=f"Classify text as positive or negative using {SENTIMENT_MODEL}.",
        examples=[
            ["This movie was absolutely fantastic! Best film of the year."],
            ["Terrible waste of time. The plot made no sense at all."],
            ["It was okay, nothing special but not bad either."],
        ],
    )
    sentiment_demo.launch()
else:
    # Demonstrate the function without Gradio
    test_texts = [
        "This movie was absolutely fantastic!",
        "Terrible waste of time.",
        "It was okay, nothing special.",
    ]
    print("=== SENTIMENT ANALYSIS (without Gradio) ===")
    for text in test_texts:
        result = analyze_sentiment(text)
        print(f"  '{text}' -> {result}")

## Part 3: Building an Image Classification Demo

Gradio's `gr.Image` component handles image uploads, webcam capture, and URL loading. Combined with a vision model, this creates a powerful image classification tool. The `gr.Label` output shows the top predicted classes with confidence scores.

In [ ]:
# Load image classification model
IMAGE_MODEL = "google/vit-base-patch16-224"

try:
    image_classifier = pipeline("image-classification", model=IMAGE_MODEL, device=-1)
    IMAGE_MODEL_LOADED = True
    print(f"Loaded: {IMAGE_MODEL}")
except Exception as e:
    IMAGE_MODEL_LOADED = False
    print(f"Could not load image model: {e}")

In [ ]:
# Build image classification demo
from PIL import Image


def classify_image(image: Image.Image) -> dict[str, float]:
    """Classify an image and return top predicted labels.

    Args:
        image: PIL Image to classify.

    Returns:
        Dictionary mapping class labels to confidence scores.
    """
    if image is None:
        return {}
    results = image_classifier(image)
    return {r['label']: r['score'] for r in results[:5]}


if GRADIO_AVAILABLE and IMAGE_MODEL_LOADED:
    image_demo = gr.Interface(
        fn=classify_image,
        inputs=gr.Image(type="pil", label="Upload an Image"),
        outputs=gr.Label(label="Predictions", num_top_classes=5),
        title="Image Classification Demo",
        description=f"Classify images into 1000 ImageNet categories using {IMAGE_MODEL}.",
    )
    image_demo.launch()
elif not GRADIO_AVAILABLE:
    print("Skipping: Gradio not available.")
else:
    print("Skipping: Image model not loaded.")

## Part 4: Custom Layouts with Gradio Blocks

`gr.Interface` is great for simple demos, but `gr.Blocks` gives you full control over layout. You can create tabbed interfaces, multi-step workflows, side-by-side comparisons, and custom event handling. This is the approach used by most production Gradio apps.

In [ ]:
# Build a multi-tab demo with Blocks
def generate_text(prompt: str, max_length: int) -> str:
    """Generate text from a prompt (simplified for demo).

    Args:
        prompt: Input text prompt.
        max_length: Maximum character length for the response.

    Returns:
        Generated or echoed text.
    """
    # Using a simple echo for demonstration (no heavy model needed)
    return f"[Generated from prompt] {prompt}" + "..." * (max_length // 20)


if GRADIO_AVAILABLE:
    with gr.Blocks(title="Multi-Tool NLP Demo") as blocks_demo:
        gr.Markdown("# Multi-Tool NLP Demo")
        gr.Markdown("Choose a tool from the tabs below.")

        with gr.Tabs():
            # Tab 1: Sentiment Analysis
            with gr.TabItem("Sentiment Analysis"):
                gr.Markdown("### Analyze the sentiment of text")
                with gr.Row():
                    with gr.Column():
                        sent_input = gr.Textbox(
                            label="Input Text",
                            placeholder="Type text to analyze...",
                            lines=3,
                        )
                        sent_btn = gr.Button("Analyze", variant="primary")
                    with gr.Column():
                        sent_output = gr.Label(label="Result", num_top_classes=2)

                sent_btn.click(fn=analyze_sentiment, inputs=sent_input, outputs=sent_output)

            # Tab 2: Text Generation
            with gr.TabItem("Text Generation"):
                gr.Markdown("### Generate text from a prompt")
                with gr.Row():
                    with gr.Column():
                        gen_input = gr.Textbox(
                            label="Prompt",
                            placeholder="Start your text...",
                            lines=2,
                        )
                        gen_slider = gr.Slider(
                            minimum=20, maximum=200, value=100,
                            step=10, label="Max Length",
                        )
                        gen_btn = gr.Button("Generate", variant="primary")
                    with gr.Column():
                        gen_output = gr.Textbox(label="Generated Text", lines=5)

                gen_btn.click(
                    fn=generate_text,
                    inputs=[gen_input, gen_slider],
                    outputs=gen_output,
                )

    blocks_demo.launch()
else:
    print("Skipping: Gradio not available.")

### Blocks Features

The Blocks API provides several layout and interaction features that aren't available with the simpler Interface API. Understanding these gives you the flexibility to build any kind of ML demo.

In [ ]:
if GRADIO_AVAILABLE:
    import pandas as pd

    features = {
        "Feature": [
            "gr.Row()", "gr.Column()", "gr.Tabs() / gr.TabItem()",
            "gr.Accordion()", "gr.Button.click()", "gr.State()",
            "gr.Examples()", "gr.Markdown()",
        ],
        "Purpose": [
            "Horizontal layout", "Vertical layout", "Tabbed interface",
            "Collapsible section", "Event handler on click", "Persistent session state",
            "Clickable example inputs", "Rich text in the UI",
        ],
        "Use Case": [
            "Side-by-side input/output", "Stacked form fields", "Multiple tools in one app",
            "Advanced settings panel", "Custom submit button", "Chat history, counters",
            "Pre-filled demo inputs", "Instructions, headers, formatting",
        ],
    }

    print("=== GRADIO BLOCKS FEATURES ===")
    print(pd.DataFrame(features).to_string(index=False))
else:
    print("Skipping: Gradio not available.")

## Part 5: Flagging, State, and Interactivity

Gradio includes built-in flagging (letting users mark problematic outputs), session state (maintaining data across interactions), and event chaining (connecting multiple actions). These features are essential for collecting feedback and building multi-step workflows.

In [ ]:
# Demo with state: a simple chat-like interface
def echo_chat(message: str, history: list[list[str]]) -> tuple[str, list[list[str]]]:
    """Simple echo chat that maintains conversation history.

    Args:
        message: The user's message.
        history: List of [user_msg, bot_msg] pairs.

    Returns:
        Tuple of (empty string for input clearing, updated history).
    """
    # Analyze sentiment of the message
    sentiment = sentiment_classifier(message)[0]
    bot_response = f"I sense this is {sentiment['label'].lower()} (confidence: {sentiment['score']:.2f}). You said: {message}"
    history.append([message, bot_response])
    return "", history


if GRADIO_AVAILABLE:
    with gr.Blocks(title="Sentiment Chat") as chat_demo:
        gr.Markdown("# Sentiment-Aware Chat")
        gr.Markdown("Type a message and the bot will analyze its sentiment.")

        chatbot = gr.Chatbot(label="Conversation")
        msg_input = gr.Textbox(
            label="Your Message",
            placeholder="Type something...",
        )
        clear_btn = gr.Button("Clear Conversation")

        msg_input.submit(echo_chat, inputs=[msg_input, chatbot], outputs=[msg_input, chatbot])
        clear_btn.click(lambda: (None, []), outputs=[msg_input, chatbot])

    chat_demo.launch()
else:
    print("Skipping: Gradio not available.")
    # Show the function working without Gradio
    print("\n=== ECHO CHAT (without Gradio) ===")
    history = []
    for msg in ["I love this tutorial!", "This is confusing."]:
        _, history = echo_chat(msg, history)
        print(f"User: {history[-1][0]}")
        print(f"Bot:  {history[-1][1]}\n")

## Part 6: Deploying to HuggingFace Spaces

HuggingFace Spaces provides free hosting for Gradio (and Streamlit) apps. Once deployed, your demo gets a permanent URL that anyone can access. This is the standard way to share ML demos in the HuggingFace ecosystem.

### 6.1 Creating a Space

To deploy a Gradio demo to HuggingFace Spaces:

**Step 1:** Create a new Space at [huggingface.co/new-space](https://huggingface.co/new-space)
- Choose a name
- Select "Gradio" as the SDK
- Choose visibility (public or private)

**Step 2:** Create an `app.py` file with your Gradio code

**Step 3:** Create a `requirements.txt` with your dependencies

**Step 4:** Push to the Space's git repository

The Space will automatically build and deploy your app.

In [ ]:
# Example app.py for a HuggingFace Space
space_app_code = '''
# app.py - Sentiment Analysis Space
import gradio as gr
from transformers import pipeline

# Load model
classifier = pipeline("sentiment-analysis",
                      model="distilbert-base-uncased-finetuned-sst-2-english")

def predict(text):
    result = classifier(text)[0]
    label = result["label"]
    score = result["score"]
    if label == "POSITIVE":
        return {"POSITIVE": score, "NEGATIVE": 1 - score}
    return {"NEGATIVE": score, "POSITIVE": 1 - score}

demo = gr.Interface(
    fn=predict,
    inputs=gr.Textbox(label="Text", lines=3),
    outputs=gr.Label(num_top_classes=2),
    title="Sentiment Analysis",
    examples=[["This is wonderful!"], ["This is terrible."]],
)

demo.launch()
'''

space_requirements = '''transformers
torch
gradio
'''

print("=== app.py ===")
print(space_app_code)

print("=== requirements.txt ===")
print(space_requirements)

print("=== DEPLOYMENT COMMANDS ===")
print("# Clone your Space repository")
print("git clone https://huggingface.co/spaces/YOUR_USERNAME/YOUR_SPACE_NAME")
print("cd YOUR_SPACE_NAME")
print("")
print("# Add files and push")
print("git add app.py requirements.txt")
print('git commit -m "Initial Space deployment"')
print("git push")

### 6.2 Space Configuration

Spaces can be configured with a `README.md` header that specifies the SDK, hardware, and other settings. Here's a typical configuration for a Gradio Space.

In [ ]:
space_readme = '''---
title: Sentiment Analysis
emoji: 🎭
colorFrom: blue
colorTo: green
sdk: gradio
sdk_version: 4.0.0
app_file: app.py
pinned: false
license: mit
---

# Sentiment Analysis Demo

Classify text as positive or negative using DistilBERT.
'''

print("=== Space README.md ===")
print(space_readme)

print("=== HARDWARE OPTIONS ===")
import pandas as pd
hardware_df = pd.DataFrame({
    "Tier": ["Free", "CPU Upgrade", "T4 Small", "T4 Medium", "A10G Small"],
    "Hardware": ["2 vCPU, 16GB RAM", "8 vCPU, 32GB RAM", "T4 GPU, 16GB", "T4 GPU, 32GB", "A10G GPU, 24GB"],
    "Cost": ["Free", "$0.03/hr", "$0.60/hr", "$0.90/hr", "$1.05/hr"],
    "Best For": ["Small models", "CPU-heavy tasks", "Medium models", "Larger models", "Large models"],
})
print(hardware_df.to_string(index=False))

## Part 7: Gradio Best Practices

Building effective ML demos requires thinking about both the user experience and the technical constraints. Here are key practices to follow when creating Gradio apps.

In [ ]:
import pandas as pd

best_practices = {
    "Category": [
        "Performance", "Performance", "Performance",
        "UX", "UX", "UX",
        "Reliability", "Reliability", "Reliability",
    ],
    "Practice": [
        "Load models outside the predict function",
        "Use batch processing for multiple inputs",
        "Cache expensive computations",
        "Provide example inputs",
        "Add clear labels and descriptions",
        "Set appropriate input constraints",
        "Handle empty/invalid inputs gracefully",
        "Add error messages users can understand",
        "Set reasonable timeouts",
    ],
    "Why": [
        "Avoids reloading the model on every request",
        "Improves throughput for multiple users",
        "Repeated inputs return instantly",
        "Users see what valid input looks like",
        "Reduces confusion and support requests",
        "Prevents OOM errors from very long inputs",
        "Prevents crashes from unexpected input",
        "Users can fix issues without developer help",
        "Prevents infinite hangs on failed inference",
    ],
}

print("=== GRADIO BEST PRACTICES ===")
print(pd.DataFrame(best_practices).to_string(index=False))

## Exercises

1. **Text Summarization Demo**: Build a Gradio demo for text summarization using `pipeline('summarization')`. Add a slider for max summary length.

2. **Image + Text Demo**: Build a demo that accepts both an image and a text question, then displays a placeholder answer (simulating a visual QA system).

3. **Model Comparison**: Create a Blocks app with two columns that runs the same input through two different sentiment models and shows results side-by-side.

4. **Deploy to Spaces**: Create a HuggingFace Space with one of your demos. Share the URL with a classmate.

5. **Custom Theme**: Explore `gr.themes` to customize the look of your demo. Try `gr.themes.Soft()` or `gr.themes.Glass()`.

In [ ]:
# Try the exercises above to build your own interactive ML demos.
# For example, create a summarization demo:
#   summarizer = pipeline('summarization', model='sshleifer/distilbart-cnn-12-6')
#   demo = gr.Interface(fn=lambda text: summarizer(text)[0]['summary_text'],
#                       inputs=gr.Textbox(lines=5), outputs=gr.Textbox())
#   demo.launch()

## Key Takeaways

- **`gr.Interface`** creates simple demos with one function, input, and output
- **`gr.Blocks`** provides full layout control with rows, columns, and tabs
- **Gradio handles data conversion** between UI components and Python types automatically
- **Examples and labels** make demos intuitive for non-technical users
- **HuggingFace Spaces** provides free hosting for permanent, shareable demos

## Next Steps

- Try **Notebook 15**: Quantization and Model Compression
- Explore the [Gradio documentation](https://gradio.app/docs/)
- Browse demos on [HuggingFace Spaces](https://huggingface.co/spaces)

## Resources

- [Gradio Quickstart](https://gradio.app/quickstart/)
- [Gradio Components](https://gradio.app/docs/components/)
- [HuggingFace Spaces Documentation](https://huggingface.co/docs/hub/spaces)
- [Gradio + HuggingFace Integration](https://gradio.app/using-hugging-face-integrations/)